In [ ]:
import copy

import torch
import torch.nn as nn
import torchvision
import numpy as np
from matplotlib import pyplot as plt

from experiment_thesis.dataset_preperation.transformation import get_transformation_sequence_images
from search.parallel_gradient import ParallelGradientDescent
from utils.sampling import BatchNegativeSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#look for experiment files in parents
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset ="bigger_emnist"# "bigger_emnist"
architecture ="bigger_extended_resnet_small"# "bigger_extended_resnet_small"
budget = 120

In [ ]:
from experiment_thesis.dataset_preperation.get_dataset import get_dataset_info,get_dataset

dataset_info = get_dataset_info(dataset)
dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

In [ ]:


dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

In [ ]:
batch_size = next(iter(train_loader))[0].shape[0]


In [ ]:
from utils.eval.vis import vis_dataset

vis_dataset(train_loader,val_loader,test_loader_transformed)

In [ ]:
from experiment_thesis.main import train_and_get_model,train_or_load_energy_model
from experiment_thesis.dataset_preperation.basic_networks import get_network
from utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "comparision_over_budget")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path,transform_name, f"{safe}.json")

In [ ]:
model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "16-mixed",
},load_if_exists=True)



In [ ]:
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")


In [ ]:
from experiment_thesis.dataset_preperation.get_dataset import get_transformation_sequence_images

In [ ]:
transform_seq = get_transformation_sequence_images(dataset_info.transform_seq_name).cuda()

In [ ]:
from search.shgo import SHGO
random_search  = SHGO(initial_samples=60)

In [ ]:
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence

In [ ]:
from torch.utils.data import SequentialSampler
from embedding_cache import LayerEmbeddingCache


from experiment_thesis.dataset_preperation.basic_networks import get_network_layer

In [ ]:


layer,layer_io = get_network_layer(dataset_info, architecture, 1, num_classes=None)

model.eval().cuda()

In [ ]:
model.eval().cuda()

In [ ]:
random_search = SHGO(initial_samples=120, local_runs=1, local_max_steps=0,project_param=True)


In [ ]:
from utils.eval.ood_performance import evaluate_confidence_and_search


In [ ]:

from embedding_cache import LayerEmbeddingCache

cache_name_train = f"{dataset}_{architecture}_{transform_name}_embedding_cache_train"

from torch.utils.data import SequentialSampler
import embedding_cache
import importlib

importlib.reload(embedding_cache)
from embedding_cache import LayerEmbeddingCache
from confidence.input_transform import RandomProjectionModule

transform_name = dataset_info.transform_seq_name

cache_name_train = f"{dataset}_{architecture}_{transform_name}_embedding_cache_train"
from experiment_thesis.dataset_preperation.get_dataset import get_layer_embedding_cache_config, \
    create_layer_embedding_cache

cache_config = get_layer_embedding_cache_config(dataset, architecture, transform_name=None, dataset_info=dataset_info)
cache_config
train_cache = create_layer_embedding_cache(model, train_loader_no_shuffle, cache_config, embedding_cache_path,
                                           device=device)

In [ ]:
from utils.transformation_problem import TransformationProblem
from confidence.direct.logit_based import EnergyConfidence

energy = SinglePassConfidence(model, EnergyConfidence())

problem_energy = TransformationProblem(energy, transform_seq, consolidate_method="consolidate_simple")

In [ ]:
test_loader_transformed = torch.utils.data.DataLoader(
    test_loader_transformed.dataset,
    batch_size=dataset_info.batch_size,
    num_workers=4,
    persistent_workers=False,
    pin_memory=True,
)

In [ ]:

evaluate_confidence_and_search(model, random_search, problem_energy, test_loader_transformed
                                             ,max_batch_override=1280)

In [ ]:
from confidence.control.split import PredictedSplitConfidence

layer_index = 4
reducer_name = "rp"
layer, layer_io = get_network_layer(dataset_info, architecture, layer_index)
embeddings_t, _, classes_t = train_cache(
            layer, capture_modes=layer_io, flatten=True, return_y=True, return_final=True, reducer_select=reducer_name
)
#print amount before
print(f"Using {embeddings_t.shape[0]} embeddings for KNN fitting")
#subsample for memory reasons
subsample = 100000
if embeddings_t.shape[0]>subsample:
    perm = torch.randperm(embeddings_t.shape[0], device=embeddings_t.device)[:subsample]
    embeddings_t = embeddings_t[perm]
    classes_t = classes_t[perm]
embeddings_t = embeddings_t.to(torch.float16)

    # 2. Build and fit detector
dtype_map = {"float32": torch.float32, "float16": torch.float16}
knn_detector = KNNConfidence(
        k=50,
        metric="cosine",
        dtype=torch.float16,
)

    #get device from kwargs
knn_detector.to(device)
knn_detector.fit(embeddings_t, classes_t)
knn_detector.to(device)

# 3. Create the full confidence module structure
dual_output_model = train_cache.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True,reducer_select=reducer_name)
conf_split = PredictedSplitConfidence(knn_detector, EnergyConfidence(), mult=False, b=0.0)
conf_mod = SinglePassConfidence(dual_output_model, conf_split, index=1)

problem = TransformationProblem(conf_mod, transform_seq, consolidate_method="consolidate_simple")

evaluate_confidence_and_search(model, random_search, problem, test_loader_transformed
                                             ,max_batch_override=1280)

In [ ]:
dual_output_model.feature_reducers

In [ ]:
train_cache.reducer_name